In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

import pyarrow.parquet as pq

acc = {}
for f in sorted(glob.glob(f"{SLIM_DIR}/*_slim.parquet")):
    names = pq.ParquetFile(f).schema.names
    genes = [g for g in GRMS if g in names]
    d = pd.read_parquet(f, columns=["class", "subclass"] + genes)
    g = d.groupby(["class", "subclass"])
    s = g[genes].sum()
    s["n"] = g.size()
    acc[f] = s
tot = pd.concat(acc.values()).groupby(level=[0, 1]).sum()
n = tot.pop("n")
mean = tot.div(n, axis=0)
mean["n_cells"] = n
mean = mean.reset_index()
mean.to_csv(f"{RES_DIR}/subclass_grm_mean_log2.csv", index=False)
m = mean[mean["n_cells"] >= MIN_CELLS].copy()
m["cn"] = m["class"].map(class_num)
m = m.sort_values(["cn", "subclass"]).reset_index(drop=True)
genes = [g for g in GRMS if g in m.columns]
fig = plt.figure(figsize=(7.5, max(6, 0.1 * len(m) + 1.5)))
ax = fig.add_axes([0.28, 0.04, 0.34, 0.93])
strip = fig.add_axes([0.255, 0.04, 0.02, 0.93])
im = ax.imshow(m[genes].to_numpy(), aspect="auto", cmap="viridis", vmin=0, interpolation="nearest")
ax.set_xticks(range(len(genes)))
ax.set_xticklabels(genes, rotation=90, fontsize=8)
ax.set_yticks(range(len(m)))
ax.set_yticklabels(m["subclass"], fontsize=3.5)
ax.yaxis.tick_right()
strip.imshow(np.array([class_color(c) for c in m["class"]])[:, None, :], aspect="auto", interpolation="nearest")
strip.set_xticks([])
strip.set_yticks([])
for cnum, grp in m.groupby("cn", sort=True):
    mid = grp.index.to_numpy().mean()
    fig.text(0.25, 0.04 + 0.93 * (1 - (mid + 0.5) / len(m)), grp["class"].iloc[0], fontsize=5, ha="right", va="center")
cax = fig.add_axes([0.66, 0.9, 0.012, 0.07])
fig.colorbar(im, cax=cax).ax.tick_params(labelsize=6)
fig.text(0.66, 0.975, "mean log2(CPM+1)", fontsize=6)
fig.savefig(f"{FIG_DIR}/ALL_fig4_heatmap_subclass_grm.png", dpi=200, bbox_inches="tight")
plt.show()